#### Feature Engineering

##### Importing Libraries and Loading Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
processed_dir = project_root /'data'/'processed'

long = pd.read_parquet(processed_dir/'station_time.parquet')
station_dim = pd.read_parquet(processed_dir /'station_dimension.parquet')

In [2]:
qcols = [c for c in long['quarter_hour'].unique()]
qcols = sorted(qcols, key=lambda c: (c[:2] < '05', c))

In [3]:
# Making a Pivot Table Frame in Chronological Time Order

wide = long.pivot_table(
    index=['nlc', 'daytype', 'metric'],
    columns='quarter_hour',
    values='count',
)

In [4]:
wide = wide[qcols]

In [5]:
wide.head()

quarter_hour         0500-0515  0515-0530  0530-0545  0545-0600  0600-0615  \
nlc daytype metric                                                           
500 fri     entries  11.125288  18.511613  28.829019  41.701635  54.004743   
            exits     2.320267   8.407356  12.947226  22.672407  39.956265   
    mon     entries   7.657625  14.688244  26.307872  40.167666  55.685555   
            exits     0.525849   6.062292   9.310302  22.566779  40.963425   
    sat     entries   7.826679  11.903178  18.346930  25.201519  32.379213   

quarter_hour         0615-0630  0630-0645   0645-0700   0700-0715   0715-0730  \
nlc daytype metric                                                              
500 fri     entries  64.747046  84.092904  106.078971  128.011669  155.298812   
            exits    59.743767  77.170189   97.200499  116.894328  132.499678   
    mon     entries  72.566382  95.719089  121.125659  146.429966  182.070004   
            exits    62.026847  80.504040  103.720505  126.206181  144.940724   
    sat     entries  36.147691  42.777563   49.048225   51.751982   53.743946   

quarter_hour         ...  0230-0245  0245-0300  0300-0315  0315-0330  \
nlc daytype metric   ...                                               
500 fri     entries  ...   2.540328   2.659495   2.995062   3.551773   
            exits    ...  17.233300  14.014180  14.258489  11.349448   
    mon     entries  ...   0.000000   0.000000   0.000000   0.000000   
            exits    ...   0.000000   0.000000   0.000000   0.000000   
    sat     entries  ...   3.327380   3.036721   2.879388   3.665862   

quarter_hour         0330-0345  0345-0400  0400-0415     0415-0430  \
nlc daytype metric                                                   
500 fri     entries   3.735997   4.915143   4.882796  9.717703e+00   
            exits    11.963900  10.205119  10.379331  8.346284e+00   
    mon     entries   0.000000   0.000000   0.000000  4.817381e+00   
            exits     0.000000   0.000000   0.000000  1.381897e-07   
    sat     entries   3.237395   4.111583   4.678346  7.420842e+00   

quarter_hour            0430-0445     0445-0500  
nlc daytype metric                               
500 fri     entries  8.503073e+00  7.931928e+00  
            exits    7.545557e+00  4.355307e+00  
    mon     entries  4.817381e+00  4.817381e+00  
            exits    1.381897e-07  1.381897e-07  
    sat     entries  6.350199e+00  6.006688e+00  

[5 rows x 96 columns]

In [6]:
wide.shape

(4710, 96)

In [7]:
# Creating Time-Window Column Groups

def in_window(prefix):
    return [c for c in qcols if c[:2] in prefix]

In [8]:
early = in_window(('05', '06'))
am = in_window(('07', '08', '09'))
mid = in_window(('10', '11', '12', '13', '14', '15'))
pm = in_window(('16', '17', '18'))
eve = in_window(('19', '20', '21'))
late = in_window(('22', '23', '00', '01', '02', '03', '04'))

In [9]:
active_nlc = station_dim[station_dim['active']]['nlc']

In [10]:
# Creating a Features matrix and Calculating features per station

records = {}
for nlc in active_nlc:
    twt_entries = wide.loc[(nlc, 'twt', 'entries')]
    twt_exits = wide.loc[(nlc, 'twt', 'exits')]
    sat_entries = wide.loc[(nlc, 'sat', 'entries')]

    twt_total = twt_entries.sum()
    sat_total = sat_entries.sum()

    twt_profile = twt_entries / twt_total
    sat_profile = sat_entries / sat_total
    f = {}

    # Log transforming the total because there is a big size difference between stations
    f['log_total_weekday'] = np.log1p(twt_total)

    f['early_share'] = twt_profile[early].sum()
    f['eve_share'] = twt_profile[eve].sum()
    f['late_share'] = twt_profile[late].sum()

    f['peakness'] = twt_profile.max()

    am_e, am_x = twt_entries[am].sum(), twt_exits[am].sum()
    pm_e, pm_x = twt_entries[pm].sum(), twt_exits[pm].sum()

    f['am_asym'] = (am_e - am_x) / (am_e + am_x)
    f['pm_asym'] = (pm_e - pm_x) / (pm_e + pm_x)

    f['weekend_shift'] = np.abs(twt_profile - sat_profile).sum() / 2
    f['weekend_ratio'] = sat_total / twt_total

    records[nlc] = f

features = pd.DataFrame(records).T
features.index.name = 'nlc'

In [11]:
features.head()

,log_total_weekday,early_share,eve_share,late_share,peakness,am_asym,pm_asym,weekend_shift,weekend_ratio
nlc,,,,,,,,,
750,7.760586,0.077621,0.094472,0.041111,0.030217,0.139383,-0.006690,0.213420,0.783216
1404,7.923830,0.071789,0.080898,0.027622,0.036185,0.116857,-0.139332,0.267944,0.725684
3000,8.666488,0.072965,0.052739,0.012879,0.043083,0.420971,-0.182355,0.291676,0.847376
500,9.014619,0.054701,0.059517,0.020899,0.040084,0.274624,-0.098968,0.242156,0.812725
502,9.483445,0.018862,0.125731,0.043576,0.049968,-0.556020,0.334524,0.255896,0.527519


In [12]:
features.shape

(432, 9)

In [13]:
# Saving DataFrame
features.to_parquet(processed_dir/'station_features.parquet')

In [14]:
features.head()

,log_total_weekday,early_share,eve_share,late_share,peakness,am_asym,pm_asym,weekend_shift,weekend_ratio
nlc,,,,,,,,,
750,7.760586,0.077621,0.094472,0.041111,0.030217,0.139383,-0.006690,0.213420,0.783216
1404,7.923830,0.071789,0.080898,0.027622,0.036185,0.116857,-0.139332,0.267944,0.725684
3000,8.666488,0.072965,0.052739,0.012879,0.043083,0.420971,-0.182355,0.291676,0.847376
500,9.014619,0.054701,0.059517,0.020899,0.040084,0.274624,-0.098968,0.242156,0.812725
502,9.483445,0.018862,0.125731,0.043576,0.049968,-0.556020,0.334524,0.255896,0.527519


In [15]:
# Loading Station Boarders and Station Flows

mon_boarders = pd.read_excel(project_root/'data'/'raw'/'NBT24MON_outputs.xlsx', sheet_name='Station_Boarders', skiprows=2)
mon_flows = pd.read_excel(project_root/'data'/'raw'/'NBT24MON_outputs.xlsx', sheet_name='Station_Flows', skiprows=2)

In [16]:
mon_boarders.head()

,ID,NLC,ASC,Station,Mode,Line,Dir,Platform,Total,Early,...,0230-0245,0245-0300,0300-0315,0315-0330,0330-0345,0345-0400,0400-0415,0415-0430,0430-0445,0445-0500
0,111091,750,ABRd,Abbey Road,DLR,DLR,IB,DLR // IB,1482.986342,88.574721,...,0,0,0,0,0,0,0.0,0.000000,0.000000,0.000000
1,111090,750,ABRd,Abbey Road,DLR,DLR,OB,DLR // OB,730.521353,65.847222,...,0,0,0,0,0,0,0.0,0.000000,0.000000,0.000000
2,620772,1404,ACCr,Acton Central,LO,NLL,EB,LO Mildmay // EB,1783.566282,142.875711,...,0,0,0,0,0,0,0.0,0.000000,0.000000,0.000000
3,620773,1404,ACCr,Acton Central,LO,NLL,WB,LO Mildmay // WB,831.360212,64.924111,...,0,0,0,0,0,0,0.0,0.000000,0.000000,0.000000
4,620816,500,ACTu,Acton Town,LU,DIS,EB,District // EB,2740.197660,214.475602,...,0,0,0,0,0,0,0.0,2.392525,2.392525,2.392525


In [17]:
mon_flows.head()

,ID,Complex NLC,From NLC,From ASC,From Station,From Node,To NLC,To ASC,To Station,To Node,...,0230-0245,0245-0300,0300-0315,0315-0330,0330-0345,0345-0400,0400-0415,0415-0430,0430-0445,0445-0500
0,620801>620816,500,500,ACTu,Acton Town,EntEx,500,ACTu,Acton Town,District // EB,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.392525e+00,2.392525e+00,2.392525e+00
1,620801>620817,500,500,ACTu,Acton Town,EntEx,500,ACTu,Acton Town,District // WB,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
2,620801>620830,500,500,ACTu,Acton Town,EntEx,500,ACTu,Acton Town,Piccadilly // EB,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
3,620801>620831,500,500,ACTu,Acton Town,EntEx,500,ACTu,Acton Town,Piccadilly // WB,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.424856e+00,2.424856e+00,2.424856e+00
4,620816>620801,500,500,ACTu,Acton Town,District // EB,500,ACTu,Acton Town,EntEx,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.381897e-07,1.381897e-07,1.381897e-07


In [18]:
mon_boarders.shape, mon_flows.shape

((1166, 111), (5599, 114))

In [19]:
# Lines Served

lines_served = mon_boarders.groupby('NLC')['Line'].nunique()

In [20]:
lines_served.name = 'lines_served'

In [21]:
lines_served

NLC
500     2
501     2
502     2
503     2
504     1
       ..
9498    1
9499    1
9586    1
9587    1
9846    1
Name: lines_served, Length: 471, dtype: int64

In [22]:
# Interchange Ratio

mon_flows['is_interchange'] = mon_flows['Movement'].str.contains("Interchange")
mon_flows['is_enter'] = mon_flows['Movement'].str.contains('Enter-Board')

C:\Users\Abdul Qudus\AppData\Local\Temp\ipykernel_17496\893443616.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mon_flows['is_interchange'] = mon_flows['Movement'].str.contains("Interchange")
C:\Users\Abdul Qudus\AppData\Local\Temp\ipykernel_17496\893443616.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mon_flows['is_enter'] = mon_flows['Movement'].str.contains('Enter-Board')


In [23]:
mon_flows['is_interchange'].value_counts()

is_interchange
True     3099
False    2500
Name: count, dtype: int64

In [24]:
mon_flows['is_enter'].value_counts()

is_enter
False    3133
True     2466
Name: count, dtype: int64

In [25]:
interchange_vol = mon_flows[mon_flows['is_interchange']].groupby('Complex NLC')['Total'].sum()
enter_vol       = mon_flows[mon_flows['is_enter']].groupby('Complex NLC')['Total'].sum()

interchange_vol = interchange_vol.reindex(active_nlc, fill_value=0)
enter_vol       = enter_vol.reindex(active_nlc, fill_value=0)

interchange_ratio = interchange_vol / (interchange_vol + enter_vol)
interchange_ratio.name = 'interchange_ratio'

In [26]:
# Adding these features to DataFrame

features = features.join(lines_served).join(interchange_ratio)

In [27]:
# Saving DataFrame with new features

features.to_parquet(processed_dir/'station_features.parquet')

In [28]:
features.head()

,log_total_weekday,early_share,eve_share,late_share,peakness,am_asym,pm_asym,weekend_shift,weekend_ratio,lines_served,interchange_ratio
nlc,,,,,,,,,,,
750,7.760586,0.077621,0.094472,0.041111,0.030217,0.139383,-0.006690,0.213420,0.783216,1,0.000000
1404,7.923830,0.071789,0.080898,0.027622,0.036185,0.116857,-0.139332,0.267944,0.725684,1,0.000000
3000,8.666488,0.072965,0.052739,0.012879,0.043083,0.420971,-0.182355,0.291676,0.847376,1,0.000000
500,9.014619,0.054701,0.059517,0.020899,0.040084,0.274624,-0.098968,0.242156,0.812725,2,0.609405
502,9.483445,0.018862,0.125731,0.043576,0.049968,-0.556020,0.334524,0.255896,0.527519,2,0.025067


In [29]:
features.shape

(432, 11)

#### Summary

This notebook turns time-series data into a features matrix: one row per active station, and a small set of columns that each summarise one aspect of how the station behaves.

We have separated shape and size of the stations. Passenger counts for big stations can be almost 500 times bigger than the smaller stations. If we fed raw counts to a clustering algorithm it would essentially sort stations by size and ignore their behavior. So most features here describe how a station is used, when it is busy, which direction people flow,
how its week changes and how connected it is largely independently of raw volume.

Every station has been given a behavioral aspect. The features come from two sources: the gateline data (entries and exits), and two network features drawn from the Station_Boarders and Station_Flows sheets.

The behavioral features are made from the TWT daytype (typical Tuesday–Thursday), because it is the most representative weekday. Mondays and Fridays are quieter due to hybrid working and other reasons, and Saturday is used as the weekend comparison.

This is what all the features are and what they mean:

- **`log_total_weekday`** is the log of total weekday entries. Logged because of station size differences. 5.7 is about 300 entries a day and 11.9 is about 147,000 entries a day.

- **`early_share`, `eve_share`, `late_share`** are the share of the day's entries falling in the early (05:00–07:00), evening (19:00–22:00) and late (22:00–5:00) time slots. These show us the character of a station. For example, early starters travel after work, and night life activity. A value of 0.07 means that 7% of people travel in that time slot during the whole day. It is the percentage of the day's entries falling in that window.

- **`peakness`** is the height of the single busiest 15 minute slice and it measures how rush-driven the station is. 0.04 means 4% of all entries happen in that peak 15 minute window.

- **`am_asym`, `pm_asym`** is the entries vs exits balance at the morning (07:00–10:00) and evening (16:00–19:00) peaks. The value runs from −1 to +1: near +1 means people mostly enter (a residential origin, people leaving home), near −1 means people mostly exit (a workplace destination, people arriving), near 0 means balanced.

- **`weekend_shift`** is how much the station's daily shape changes from weekday to
  Saturday. 0 = identical, 1 = completely different. Shows stations with changing patterns: sharp commuter peaks on weekdays that change into leisurely Saturday use.

- **`weekend_ratio`** is Saturday's total entries divided by weekdays' total entries.  0.79 means Saturday is 79% as busy as a weekday. Values above 1 mean the station is busier on Saturday than midweek, so it can exceed 100%.

- **`lines_served`** is the number of distinct lines serving the station, self explanatory.

- **`interchange_ratio`** is all the activity that starts at a station (people
  entering from the street to board, plus people changing trains). Near 0 = an origin/destination station and near 1 = a pure
  transfer hub like West Ham or Stockwell, where most movement is people changing trains
  without ever touching a gateline.